# AI301 Assignment #2 — Soft-voting ensemble of the CNN runs

**NEW notebook (AI-assisted, Claude), written for the final code** (`eff_net_augumented.ipynb` = `b4_sceneval_aug`, the earlier no-augmentation `b4_sceneval`, and the ConvNeXt runs of `convnext_augumented.ipynb`). Cells marked *copied unchanged* are taken 1:1 from `eff_net_on_cleaned_data.ipynb`, so thresholds and the species rule are computed exactly as in the single-model runs.

**What it does.** Every training run of `eff_net_on_cleaned_data.ipynb` saves two files (flip-TTA probabilities of its best epoch):
- `val_probs_<run_tag>.npy`: the val images (`is_val`) in `train_clean.csv` order, shape `(n_val, 5)`
- `test_probs_<run_tag>.npy`: the test images in `test.csv` order, shape `(18673, 5)`

**Soft voting** = average the predicted *probabilities* of the models (hard voting would average 0/1 decisions and lose the confidence). Then, as for a single model:
tune one threshold per class on the averaged val probabilities (rare classes first, then the species rule, then Cat / Dog) → apply them to the averaged test probabilities.
This is **ensembling, not stacking**: no second-level model is trained.

**Rules to keep the result honest**
- All runs must come from the **same `train_clean.csv` split** (`is_val`). The very first EfficientNet run (`assignment-2-dl-sample-submission_eff_net.ipynb`: random stratified split, no deduplication) and the ResNet baseline must **not** be mixed in: their training sets contain images that are in the new val set.
- Decide the member list and the weighting **before** looking at the table. Default: all runs, with **family-equal** weights, i.e. the EfficientNet runs together weigh as much as the ConvNeXt runs together (otherwise two B4 runs would outvote the single ConvNeXt 2:1). Picking the best-looking subset on ~8–11 rare val positives is selection bias; the table is for the report.
- The tuned val F1 is optimistic (thresholds are fitted on the same val images). Compare models with the untuned `F1_t05` and the AP columns.

**Kaggle setup:** attach the competition data, the `train_clean.csv` dataset and the outputs of the training runs (the notebooks' output folders as datasets).

## 1. Setup & inputs

In [ ]:
import os, glob, time, warnings, itertools
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, average_precision_score

warnings.filterwarnings("ignore")
SEED = 42
ON_KAGGLE = os.path.exists("/kaggle/input")
OUT_DIR = Path("/kaggle/working") if ON_KAGGLE else Path("output")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# run_tag of every model that should vote (= CFG["run_tag"] of the training notebook; missing runs are skipped with a warning)
RUNS = ["b4_sceneval_aug", "b4_sceneval", "convnext_t_aug", "convnext_t_attn_aug"]
#   b4_sceneval_aug     EfficientNet-B4, albumentations on          (eff_net_augumented.ipynb)
#   b4_sceneval         EfficientNet-B4, no common augmentation     (eff_net_on_cleaned_data.ipynb / effnet_b4_scene_val.ipynb)
#   convnext_t_aug      ConvNeXt-Tiny, GAP head, same augmentations (convnext_augumented.ipynb, PRESET = "convnext_t_gap")
#   convnext_t_attn_aug ConvNeXt-Tiny, class-attention head         (convnext_augumented.ipynb, PRESET = "convnext_t_attn")
FAMILY = {r: ("convnext" if "convnext" in r else "efficientnet") for r in RUNS}   # backbone family of every run
WEIGHTING = "family_equal"   # "equal" | "family_equal" (each family gets the same total weight) | "val_mAP" (proportional to each model's val mAP)
ENSEMBLE_NAME = "softvote"   # name of the output files

In [ ]:
def find_file(name, roots):
    """First `name` under `roots`; follows symlinks (Kaggle may mount inputs through them)."""
    for root in roots:
        if not os.path.isdir(root):
            continue
        hits = glob.glob(os.path.join(root, "**", name), recursive=True)
        if not hits:
            hits = [os.path.join(r, name) for r, _, files in os.walk(root, followlinks=True) if name in files]
        if hits:
            return Path(sorted(hits)[0])
    return None

ROOTS = ["/kaggle/working", "/kaggle/input/datasets", "/kaggle/input/notebooks", "/kaggle/input", "output", "data", "."]
train_csv = find_file("train.csv", ["/kaggle/input/competitions", "/kaggle/input", "data", "."])
clean_csv = find_file("train_clean.csv", ROOTS)
assert train_csv and clean_csv, "attach the competition data and the train_clean.csv dataset"
DATA_DIR = train_csv.parent

test = pd.read_csv(DATA_DIR / "test.csv", dtype={"image_id": str})
sample_sub = pd.read_csv(DATA_DIR / "sample_submission.csv", dtype={"image_id": str})
LABELS = sample_sub.columns.drop("image_id").tolist()
RARE = ["Moris", "Motya", "Biatrix"]

# Cleaned train table, parsed exactly as in the training notebook
train = pd.read_csv(clean_csv, dtype={"image_id": str})
train["is_val"] = train.is_val.astype(str).str.lower().eq("true") if train.is_val.dtype == object else train.is_val.astype(bool)
y_val = train.loc[train.is_val, LABELS].values          # val labels, same row order as every val_probs_*.npy
print("train_clean", train.shape, "| val", y_val.shape, "| test", test.shape)
print("val positives:", dict(zip(LABELS, y_val.sum(axis=0))))

## 2. Load the saved probabilities of every run

In [ ]:
val_p, test_p = {}, {}
for run in RUNS:
    fv, ft = find_file(f"val_probs_{run}.npy", ROOTS), find_file(f"test_probs_{run}.npy", ROOTS)
    if fv is None or ft is None:
        print(f"-- skipped {run}: val_probs / test_probs not found")
        continue
    v, t = np.load(fv), np.load(ft)
    assert v.shape == y_val.shape, f"{run}: val probabilities {v.shape} != val labels {y_val.shape} (another split?)"
    assert t.shape == (len(test), len(LABELS)), f"{run}: test probabilities {t.shape} do not match test.csv"
    # order check: Cat / Dog are easy classes, so a misaligned val order shows up as a low AP
    ap_easy = [average_precision_score(y_val[:, j], v[:, j]) for j in (LABELS.index("Cat"), LABELS.index("Dog"))]
    assert min(ap_easy) > 0.9, f"{run}: Cat/Dog val AP {ap_easy} is too low: the val rows are probably in another order"
    val_p[run], test_p[run] = v, t
    print(f"loaded {run:18s} val {v.shape} test {t.shape}  <- {fv.name}, {ft.name}")
assert len(val_p) >= 2, "an ensemble needs at least two runs"
MODELS = list(val_p)

## 3. Thresholds, species rule, scores
*Copied unchanged* from `eff_net_on_cleaned_data.ipynb` (section 5): `enforce_hierarchy`, `macro_f1`, `plateau_threshold`, `tune_thresholds`. Only `score` and `bootstrap_ci` are new.

In [ ]:
# ---- copied unchanged from eff_net_on_cleaned_data.ipynb ------------------------------------------------------------
def enforce_hierarchy(pred):
    """0/1 predictions (N, 5): an identity label switches on its species label."""
    pred = pred.copy()
    col = {lab: j for j, lab in enumerate(LABELS)}
    pred[:, col["Cat"]] |= pred[:, col["Moris"]] | pred[:, col["Motya"]]
    pred[:, col["Dog"]] |= pred[:, col["Biatrix"]]
    return pred

def macro_f1(y, pred):
    return f1_score(y, pred, average="macro", zero_division=0)

GRID = np.linspace(0.01, 0.99, 99)

def plateau_threshold(f1s):
    """Middle of the longest run of thresholds that reach the best F1."""
    idx = np.flatnonzero(f1s >= f1s.max() - 1e-9)
    runs = np.split(idx, np.where(np.diff(idx) > 1)[0] + 1)
    run_ = max(runs, key=len)
    return float(GRID[run_[len(run_) // 2]])

def tune_thresholds(y, p):
    """Rare classes first, then Cat / Dog with the species rule applied. Returns thresholds (5,) and F1 curves."""
    col = {lab: j for j, lab in enumerate(LABELS)}
    th, curves = np.full(len(LABELS), 0.5), {}
    for lab in RARE:
        j = col[lab]
        curves[lab] = np.array([f1_score(y[:, j], (p[:, j] > t).astype(int), zero_division=0) for t in GRID])
        th[j] = plateau_threshold(curves[lab])
    rare_pred = {lab: (p[:, col[lab]] > th[col[lab]]).astype(int) for lab in RARE}
    for species, pets in [("Cat", ["Moris", "Motya"]), ("Dog", ["Biatrix"])]:
        j = col[species]
        forced = np.max([rare_pred[pet] for pet in pets], axis=0)
        curves[species] = np.array([f1_score(y[:, j], (p[:, j] > t).astype(int) | forced, zero_division=0) for t in GRID])
        th[j] = plateau_threshold(curves[species])
    return th, curves
# ---- end of the copied block ---------------------------------------------------------------------------------------

# ===== NEW (AI-assisted, Claude) =====
def score(y, p):
    """Val scores of probabilities p (n, 5): threshold-free AP, F1 at 0.5 (+ species rule), F1 with tuned thresholds (optimistic)."""
    th, _ = tune_thresholds(y, p)
    pred = enforce_hierarchy((p > th).astype(int))
    ap = [average_precision_score(y[:, j], p[:, j]) for j in range(len(LABELS))]
    row = dict(mAP=np.mean(ap), **{f"AP_{lab}": a for lab, a in zip(LABELS, ap)},
               F1_t05=macro_f1(y, enforce_hierarchy((p > 0.5).astype(int))), F1_tuned=macro_f1(y, pred))
    return row, th, pred

def bootstrap_ci(y, pred, n=1000):
    """95% interval of the macro F1 when the val set is drawn again (thresholds fixed)."""
    rng = np.random.RandomState(SEED)
    boot = [macro_f1(y[b], pred[b]) for b in (rng.randint(0, len(y), len(y)) for _ in range(n))]
    return np.percentile(boot, [2.5, 97.5])

def member_weights(names, mode):
    """Weights of the members `names` (sum to 1) for mode "equal", "family_equal" or "val_mAP"."""
    if mode == "equal":
        w = {n: 1.0 for n in names}
    elif mode == "family_equal":                        # every family gets the same total weight, split equally inside the family
        n_fam = {f: sum(FAMILY[n] == f for n in names) for f in {FAMILY[n] for n in names}}
        w = {n: 1.0 / (len(n_fam) * n_fam[FAMILY[n]]) for n in names}
    elif mode == "val_mAP":
        w = {n: w_map[n] for n in names}
    else:
        raise ValueError(mode)
    total = sum(w.values())
    return {n: v / total for n, v in w.items()}

def soft_vote(probs, names, mode="equal"):
    """Soft voting: weighted mean of the probabilities of `names`."""
    w = member_weights(names, mode)
    return np.average(np.stack([probs[n] for n in names]), axis=0, weights=[w[n] for n in names])
# ===== END NEW =====

## 4. Single models vs ensembles (val)
Rows: every model alone, the soft-voting ensemble of all models (equal and mAP-weighted) and each leave-one-model-out ensemble.
`F1_tuned` has thresholds fitted on these same val images, so it is optimistic; look at `mAP`, the rare-class APs and `F1_t05` first.

In [ ]:
# ===== NEW (AI-assisted, Claude): comparison table =====
single = {m: score(y_val, val_p[m])[0] for m in MODELS}
w_map = {m: single[m]["mAP"] for m in MODELS}            # candidate weights: each model's own val mAP

variants = {m: val_p[m] for m in MODELS}
variants["ENSEMBLE all, equal"] = soft_vote(val_p, MODELS, "equal")
variants["ENSEMBLE all, family-equal"] = soft_vote(val_p, MODELS, "family_equal")
variants["ENSEMBLE all, mAP-weighted"] = soft_vote(val_p, MODELS, "val_mAP")
for fam in sorted({FAMILY[m] for m in MODELS}):                   # soft voting inside one family (needs >= 2 members)
    members = [m for m in MODELS if FAMILY[m] == fam]
    if len(members) >= 2:
        variants[f"{fam} only (equal)"] = soft_vote(val_p, members, "equal")
if len(MODELS) > 2:
    for m in MODELS:
        variants[f"ensemble without {m}"] = soft_vote(val_p, [x for x in MODELS if x != m], WEIGHTING)
ENS_KEY = {"equal": "ENSEMBLE all, equal", "family_equal": "ENSEMBLE all, family-equal", "val_mAP": "ENSEMBLE all, mAP-weighted"}[WEIGHTING]

rows = {}
for name, p in variants.items():
    row, th, pred = score(y_val, p)
    lo, hi = bootstrap_ci(y_val, pred)
    rows[name] = dict(**row, F1_tuned_ci=f"{lo:.3f}-{hi:.3f}")
table = pd.DataFrame(rows).T
cols = ["mAP", "AP_Moris", "AP_Motya", "AP_Biatrix", "F1_t05", "F1_tuned", "F1_tuned_ci"]
display(table[cols].astype({c: float for c in cols[:-1]}).round(4))
# ===== END NEW =====

### How different are the members? (correlation of rare-class probabilities on val; low correlation = more to gain from voting)

In [ ]:
# ===== NEW (AI-assisted, Claude) =====
pairs = list(itertools.combinations(MODELS, 2))
corr = pd.DataFrame({lab: [np.corrcoef(val_p[a][:, LABELS.index(lab)], val_p[b][:, LABELS.index(lab)])[0, 1] for a, b in pairs]
                     for lab in RARE}, index=[f"{a} vs {b}" for a, b in pairs])
display(corr.round(3))

ap_cols = [f"AP_{lab}" for lab in RARE]
plot_df = table.loc[[m for m in MODELS] + [ENS_KEY], ap_cols].astype(float)
plot_df.columns = RARE
ax = plot_df.plot.bar(figsize=(10, 4), rot=20)
ax.set(ylabel="val AP", title="Rare-class AP: single models vs the soft-voting ensemble")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
# ===== END NEW =====

## 5. Final ensemble: thresholds, test predictions, sanity checks, submission
The member list and the weighting are fixed at the top (`RUNS`, `WEIGHTING`). Thresholds come from the **averaged val probabilities**, as for a single model.

In [ ]:
# ===== NEW (AI-assisted, Claude) =====
print("members:", MODELS, "| weighting:", WEIGHTING, "|", {m: round(w, 3) for m, w in member_weights(MODELS, WEIGHTING).items()})

p_val_ens = soft_vote(val_p, MODELS, WEIGHTING)
p_test_ens = soft_vote(test_p, MODELS, WEIGHTING)

thresholds, f1_curves = tune_thresholds(y_val, p_val_ens)
val_row, _, val_pred = score(y_val, p_val_ens)
lo, hi = bootstrap_ci(y_val, val_pred)
res = pd.DataFrame({
    "val_positives": y_val.sum(axis=0),
    "AP": [average_precision_score(y_val[:, j], p_val_ens[:, j]) for j in range(len(LABELS))],
    "threshold": thresholds,
    "F1_tuned": [f1_score(y_val[:, j], val_pred[:, j], zero_division=0) for j in range(len(LABELS))],
}, index=LABELS).round(3)
display(res)
print(f"ensemble val macro F1 (tuned thresholds + species rule): {val_row['F1_tuned']:.4f}  [95% bootstrap {lo:.3f}-{hi:.3f}]  (optimistic)")
print(f"ensemble val macro F1 @ 0.5 + species rule: {val_row['F1_t05']:.4f} | val macro AP: {val_row['mAP']:.4f}")

# ---- test predictions + sanity checks (same checks as the single-model notebook) ----
test_pred = enforce_hierarchy((p_test_ens > thresholds).astype(int))
submission = pd.DataFrame(test_pred, columns=LABELS)
submission.insert(0, "image_id", test.image_id.values)
assert list(submission.columns) == list(sample_sub.columns)
assert (submission.image_id.values == sample_sub.image_id.values).all()

prev = train[LABELS].mean()
checks = pd.DataFrame({
    "test_pred_%": (submission[LABELS].mean() * 100).round(2),
    "train_%": (prev * 100).round(2),
    "test_pred_count": submission[LABELS].sum(),
    "expected_count": (prev * len(test)).round(0).astype(int),
})
display(checks)
col = {lab: j for j, lab in enumerate(LABELS)}
assert not ((test_pred[:, col["Moris"]] | test_pred[:, col["Motya"]]) & (1 - test_pred[:, col["Cat"]])).any(), "species rule broken"
assert not (test_pred[:, col["Biatrix"]] & (1 - test_pred[:, col["Dog"]])).any(), "species rule broken"
for lab in RARE:
    n, exp = checks.test_pred_count[lab], checks.expected_count[lab]
    if not 0.33 * exp <= n <= 3 * exp:
        print(f"WARNING {lab}: {n} predicted vs ~{exp} expected -> check its threshold ({thresholds[col[lab]]:.2f})")
blank = submission[submission.image_id.str.startswith("d20a3fd2")]
print("blank white test image:", blank.drop(columns="image_id").to_dict("records"))

out_name = f"submission_{ENSEMBLE_NAME}.csv"
submission.to_csv(OUT_DIR / out_name, index=False)
np.save(OUT_DIR / f"val_probs_{ENSEMBLE_NAME}.npy", p_val_ens)    # reuse: e.g. as a member of a later ensemble
np.save(OUT_DIR / f"test_probs_{ENSEMBLE_NAME}.npy", p_test_ens)
print("saved", out_name)
display(submission.head())
# ===== END NEW =====

### Results log
One row per ensemble in `results_log_ensemble.csv`. After submitting, fill in **public_LB**: the report needs CV (val) **and** LB for every experiment, single models and the ensemble alike.

In [ ]:
# ===== NEW (AI-assisted, Claude) =====
row = dict(
    run=ENSEMBLE_NAME, time=time.strftime("%Y-%m-%d %H:%M"), members="+".join(MODELS), weighting=WEIGHTING, weights=str({m: round(w, 3) for m, w in member_weights(MODELS, WEIGHTING).items()}),
    val_mAP=round(val_row["mAP"], 4), val_macroF1_t05=round(val_row["F1_t05"], 4), val_macroF1_tuned=round(val_row["F1_tuned"], 4),
    val_macroF1_ci_lo=round(lo, 4), val_macroF1_ci_hi=round(hi, 4),
    **{f"F1_{lab}": res.F1_tuned[lab] for lab in LABELS}, **{f"thr_{lab}": res.threshold[lab] for lab in LABELS},
    **{f"test_n_{lab}": int(checks.test_pred_count[lab]) for lab in LABELS}, submission=out_name, public_LB=np.nan,
)
log_path = OUT_DIR / "results_log_ensemble.csv"
log = pd.concat([pd.read_csv(log_path), pd.DataFrame([row])], ignore_index=True) if log_path.exists() else pd.DataFrame([row])
log.to_csv(log_path, index=False)
display(log.tail(5))
# ===== END NEW =====

### Notes for the report
- **Why soft voting helps here:** members with different backbones (EfficientNet vs ConvNeXt) and different heads (GAP vs attention pooling) make different mistakes; averaging probabilities smooths single-model errors, mainly on the rare classes where one miss is expensive.
- **Why not stacking:** a second-level model on ~8–11 rare val positives would overfit. If stacking is added anyway (optional in the assignment), the level-2 validation has to be nested: fit the meta-model on out-of-fold predictions only, and evaluate it on images no level-1 or level-2 model has seen.
- **Caveat:** thresholds and the ensemble are tuned on one small val set; compare the val score with the **Public LB**. The LB contains sibling shots of the rare pets (the val does not), so a much higher LB than val means the models lean on the scene.